# Quickstart

A basic introduction to `atlas-local-lib-py`: create a MongoDB Atlas Local
deployment, inspect its state, retrieve its connection string, and delete it
when finished.

Requires a Docker daemon on the machine running this kernel.

In [1]:
%pip install atlas-local-lib-py


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Create a deployment

`get_or_create` is the notebook-friendly entry point: re-running this cell
returns the deployment that already exists instead of failing or creating a
second one. The first run pulls the image, so it takes a while.

In [3]:
from atlas_local import LocalDeployment

NAME = "quickstart"

deployment = LocalDeployment.get_or_create(name=NAME)
deployment

LocalDeployment(name=Some("quickstart"), container_id="a13af7766c980c7874a68d03647c91340093c1e7e1ea1bc85ddca7c9c796d6e3", state="running", mongodb_version="8.3.4")

## Inspect its state

A deployment is running as soon as it is created.

In [4]:
print("name:         ", deployment.name)
print("container id: ", deployment.container_id[:12])
print("state:        ", deployment.state)
print("image:        ", deployment.image)
print("image tag:    ", deployment.image_tag)
print("mongodb:      ", deployment.mongodb_version)
print("port binding: ", deployment.port_bindings)

name:          quickstart
container id:  a13af7766c98
state:         running
image:         quay.io/mongodb/mongodb-atlas-local
image tag:     latest
mongodb:       8.3.4
port binding:  127.0.0.1/52674


It also shows up in the list of local deployments.

In [5]:
[(other.name, other.state) for other in LocalDeployment.list()]

[('quickstart', 'running'),
 ('probe-g', 'exited'),
 ('notebook-demo', 'exited'),
 ('probe-f', 'exited'),
 ('probe-a', 'exited'),
 ('probe-e', 'exited'),
 ('probe-d', 'exited'),
 ('probe-c', 'exited')]

## Get the connection string

The connection string points at the port Docker published on the host. It only exists while
the deployment is running, so a stopped or paused deployment has to be started
or unpaused first.

In [7]:
connection_string = deployment.connection_string()
connection_string

'mongodb://127.0.0.1:52674/?directConnection=true'

## Connect from Python

The connection string returned by this library can be used with any MongoDB-compatible
driver, such as `PyMongo`. `PyMongo` is not a dependency of this library, so
it must be installed separately.

In [8]:
%pip install pymongo


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pymongo

client = pymongo.MongoClient(connection_string)
client.quickstart.movies.insert_one({"title": "Local Atlas", "year": 2026})

list(client.quickstart.movies.find({}, {"_id": 0}))

[{'title': 'Local Atlas', 'year': 2026}]

## Delete the deployment

Deleting the deployment removes its container and all data stored in it. To keep it available between notebook sessions, leave this cell unexecuted. The `get_or_create` call in the first cell will reuse it.


In [10]:
client.close()
deployment.delete()

[other.name for other in LocalDeployment.list()]

['probe-g',
 'notebook-demo',
 'probe-f',
 'probe-a',
 'probe-e',
 'probe-d',
 'probe-c']